# Lab Day 10 — Đinh Quang Lâm · MSSV 202602875

Code đầy đủ cho Qwen3-VL QLoRA. Các thí nghiệm GPU **chưa chạy** trong môi trường hiện tại. Chạy trên Colab T4 để tạo evidence thật; không sửa scripts/tests. Hướng dẫn thêm: RUNNING.md.

## 0. Môi trường (Colab T4)
Chọn Runtime → Change runtime type → T4 GPU. Cell dừng ngay nếu không có CUDA. Repo và dataset đều công khai; không cần nhúng token/Kaggle key. Checkpoint có thể lưu Drive; không commit trọng số.

In [ ]:
from pathlib import Path
import os, subprocess, sys
REPO_URL = "https://github.com/qlam1208/K4-Track-4-Day10-VisionLanguageModel.git"
ROOT = Path("/content/K4-Track-4-Day10-VisionLanguageModel")
if not ROOT.exists():
    subprocess.run(["git", "clone", REPO_URL, str(ROOT)], check=True)
os.chdir(ROOT)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements-colab.txt", "pytest"], check=True)
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Please select a T4 GPU runtime before running E0–E4.")
print(torch.cuda.get_device_name(0), torch.cuda.get_device_properties(0).total_memory / 1024**3, "GB")
# Colab may hide output inherited by a subprocess: capture and print it explicitly.
# Keep failing tests fatal, and save the full diagnostic log for troubleshooting.
print("Python:", sys.version)
print("PyTorch:", torch.__version__, "CUDA:", torch.version.cuda)
print("Git revision:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())
test_result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "-ra", "--tb=short", "tests", "student_tests",
     "--deselect=tests/test_configs_and_docs.py::test_report_template_has_student_markers_and_ten_questions"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
log_path = ROOT / "work/diagnostics/startup-tests.txt"
log_path.parent.mkdir(parents=True, exist_ok=True)
log_path.write_text(test_result.stdout)
print(test_result.stdout, flush=True)
if test_result.returncode:
    raise RuntimeError(
        f"Startup tests failed (exit {test_result.returncode}). "
        f"Read the pytest failure above; full log: {log_path}. "
        "Do not proceed to E0–E4 until it is resolved.")


In [ ]:
SAVE_TO_DRIVE = True  # persistent checkpoints; first mount requires Google authorization
CHECKPOINT_ROOT = None
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    CHECKPOINT_ROOT = "/content/drive/MyDrive/K4-Day10-202602875/checkpoints"
    Path(CHECKPOINT_ROOT).mkdir(parents=True,exist_ok=True)


## 1. Dữ liệu: tải, chia session và kiểm tra nhãn
Chạy công cụ gốc với URL zip công khai. Nếu endpoint yêu cầu đăng nhập, tải zip thủ công rồi đặt DATASET_ZIP thành đường dẫn; cũng có thể dùng --kaggle với Colab Secrets. Không commit data/work.

In [ ]:
DATASET_ZIP = "https://www.kaggle.com/api/v1/datasets/download/lorenzoarcioni/road-damage-dataset-potholes-cracks-and-manholes"
if not Path("data/splits/meta.json").exists():
    subprocess.run([sys.executable,"scripts/prepare_data.py","--zip",DATASET_ZIP],check=True)
from student_vlm.inspection import inspect_data
import json
print(json.dumps(inspect_data(),indent=2))
from IPython.display import display, Image
for path in sorted(Path("report/figures").glob("gt_train_*.png")):
    display(Image(filename=str(path)))


## 2. Nhãn: YOLO → JSON thang 0–1000
Hàm thực nằm trong student_vlm.labels; unit tests kiểm tra round/clamp, thứ tự y/x và reward.

In [ ]:
from scripts.labcommon import read_jsonl
from student_vlm.labels import yolo_to_box, objects_to_answer, PROMPT
train_records = read_jsonl("data/splits/train.jsonl")
print(PROMPT)
print(objects_to_answer(train_records[0]["objects"]))
print(yolo_to_box("pothole",[0.5,0.5,0.2,0.4]))


## 3. E0: zero-shot
NF4 4-bit; visual và lm_head không lượng tử hóa. Raw không được sửa trước chấm. Cell chỉ chạy một lần trên full TEST, file đã có sẽ được bỏ qua để tránh retest.

In [ ]:
from student_vlm.experiments import config_for, evaluate_checkpoint
if not Path("report/evidence/e0.json").exists():
    evaluate_checkpoint(config_for("E0"))
print(Path("report/evidence/e0.json").read_text())
print("First five raw model outputs:")
for r in read_jsonl("work/e0_test.jsonl")[:5]:print(r)


## 4. Dữ liệu huấn luyện, prefix và mask loss
Kiểm tra 220 token ảnh, prefix và đáp án trước khi train. pixel_values/image_grid_thw nối theo chiều 0; giữ mm_token_type_ids cho M-RoPE.

In [ ]:
from transformers import AutoProcessor
from student_vlm.encoding import encode_example, open_image, collate_examples
processor = AutoProcessor.from_pretrained("Qwen/Qwen3-VL-4B-Instruct")
example = encode_example(processor,open_image(train_records[0],"data"),train_records[0]["objects"])
print("vision tokens:",int((example["input_ids"]==151655).sum()))
print("supervised answer:",processor.tokenizer.decode(example["labels"][example["labels"]!=-100]))
print({k:tuple(v.shape) for k,v in collate_examples([example],processor.tokenizer.pad_token_id).items()})


## 5. Setup LoRA / đóng băng
apply_setup kiểm tra 252/348 module và ViT unquantized; trainable parameters fp32. Bảng dưới đếm cấu trúc thật trên meta, không phải huấn luyện hay VRAM.

In [ ]:
from transformers import Qwen3VLConfig, Qwen3VLForConditionalGeneration, AutoConfig
from student_vlm.modeling import apply_setup
model_config = AutoConfig.from_pretrained("Qwen/Qwen3-VL-4B-Instruct")
for name in ("E1","E2","E3"):
    with torch.device("meta"):
        skeleton=Qwen3VLForConditionalGeneration(model_config)
        skeleton, info=apply_setup(skeleton,config_for(name))
    print(name,info["trainable_params"],len(info["matched_modules"]))
    del skeleton


## 6. Huấn luyện E1–E3
300 optimizer steps; batch 2 × accumulation 4; warmup 5%, cosine, fp16 autocast, GradScaler, clip 1. Checkpoint mỗi 100 bước: adapter + projector riêng + optimizer/scaler/scheduler/RNG. Drive chỉ giữ latest mỗi setup để tránh lưu trùng nhiều GB. Nếu ngắt, dùng train(...,resume=đường_dẫn_latest). Không đổi config khi resume.

In [ ]:
from student_vlm.training import train
checkpoints={}
for name in ("E1","E2","E3"):
    final=Path("work")/name.lower()/"step-0300"
    if not (final/"run.json").exists():
        final=train(config_for(name),checkpoint_root=CHECKPOINT_ROOT)
    checkpoints[name]=final
    print(name,json.loads((final/"run.json").read_text()))


## 7. Suy luận, chấm E1–E3 và lựa chọn bằng VAL
Nạp base + adapter + projector. VAL dùng chọn base E4; TEST chỉ tạo một bản kết quả mỗi setup. Evidence do score.py gốc sinh ra với meta lấy từ run.json thật.

In [ ]:
validation={}
for name in ("E1","E2","E3"):
    val_path=Path("work/validation")/(name.lower()+".json")
    if not val_path.exists():
        evaluate_checkpoint(config_for(name),checkpoints[name],split="val")
    validation[name]=json.loads(val_path.read_text())
    if not (Path("report/evidence")/(name.lower()+".json")).exists():
        evaluate_checkpoint(config_for(name),checkpoints[name])
    print(name, json.loads((Path("report/evidence")/(name.lower()+".json")).read_text()))


## 8. E4 rank ablation
Chọn base từ VAL, chạy cả r=4 và r=64 với alpha=2r. Mọi biến khác giữ nguyên. Không chọn lại bằng TEST. E5 tùy chọn: reward đã có test; cell này không tuyên bố đã chạy GRPO.

In [ ]:
from student_vlm.experiments import choose_ablation_base, rank_ablation
base=choose_ablation_base(validation)
choice={"base":base,"selected_on":"val","variable":"rank","values":[4,64],
        "dataset_version":json.loads(Path("data/splits/meta.json").read_text())["dataset_version"],
        "validation_f1":{n:r["micro"]["f1"] for n,r in validation.items()}}
Path("report/diagnostics/ablation_choice.json").write_text(json.dumps(choice,indent=2))
for cfg in rank_ablation(config_for(base)):
    final=Path("work")/cfg["name"].lower()/"step-0300"
    if not (final/"run.json").exists():final=train(cfg,checkpoint_root=CHECKPOINT_ROOT)
    if not (Path("report/evidence")/(cfg["name"].lower()+".json")).exists():
        evaluate_checkpoint(cfg,final)


In [ ]:
from student_vlm.labels import iou_reward
print("Perfect reward:",iou_reward('[{"label":"pothole","bbox_2d":[100,100,300,300]}]',
      [{"label":"pothole","bbox":(100,100,300,300)}]))
subprocess.run([sys.executable,"-m","pytest","-q","student_tests/test_labels.py"],check=True)


## Hoàn tất báo cáo, hình lỗi và đóng gói
Code cập nhật bảng/số, nhưng phải đọc output/ảnh và điền kết luận thực nghiệm trước nộp. Gói chỉ chứa code, evidence, hình và log nhẹ; không có data/work/key/trọng số. Push Git bằng xác thực GitHub của bạn, không ghi token vào notebook.

In [ ]:
from student_vlm.reporting import update_report,prediction_figures
update_report()
prediction_figures()
for path in sorted(Path("report/figures").glob("e1_error_*.png")):display(Image(filename=str(path)))
subprocess.run([sys.executable,"scripts/check_submission.py"],check=True)
# Copy loss/run JSON logs to report/logs; exclude all checkpoint tensors.
import shutil,zipfile
for name in ("e1","e2","e3","e4_rank_4","e4_rank_64"):
    for leaf in ("loss.jsonl","run.json"):
        src=Path("work")/name/leaf
        if src.exists():
            dst=Path("report/logs")/name/leaf
            dst.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(src,dst)
archive=Path("/content/K4-DAY10-DinhQuangLam-202602875-results.zip")
with zipfile.ZipFile(archive,"w",zipfile.ZIP_DEFLATED) as z:
    for p in Path("report").rglob("*"):
        if p.is_file() and p.suffix not in (".pt",".bin",".safetensors"):z.write(p,str(p))
    z.write("REPORT.md");z.write("lab_day10.ipynb")
if SAVE_TO_DRIVE:shutil.copy2(archive,Path(CHECKPOINT_ROOT).parent/archive.name)
from google.colab import files
files.download(str(archive))
